## 0. Imports and data loading

In this homework we will use **Fashion-MNIST** as an example dataset.
It is slightly harder than MNIST but still small enough to train quickly in Colab.

In [12]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)

# Load Fashion-MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Add channel dimension: (N, 28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

num_classes = 10
input_shape = x_train.shape[1:]

print('Train shape:', x_train.shape)
print('Test shape:', x_test.shape)
print('Input shape:', input_shape)

2.19.0
Train shape: (60000, 28, 28, 1)
Test shape: (10000, 28, 28, 1)
Input shape: (28, 28, 1)


## Task 1: Build an end-to-end CNN model

Implement the function `build_e2e_cnn(input_shape, num_classes)`.

Requirements:
- Use `tf.keras` / `keras` only.
- The model must be **end-to-end**:
  - Input: raw normalized images of shape `input_shape` (e.g., `(28, 28, 1)`).
  - Output: logits or probabilities over `num_classes` classes.
- Architecture constraints:
  - Use at least **2 convolutional layers** (`Conv2D`).
  - Use at least **1 pooling layer** (`MaxPooling2D`).
  - Use at least **1 Dense hidden layer** before the final output layer.
- The final layer must have `num_classes` units.
- The model must be **compiled** with:
  - optimizer: `'adam'`
  - loss: `'sparse_categorical_crossentropy'`
  - metric: `'accuracy'`

Feel free to change the number of filters/units, as long as the constraints above are satisfied.

In [13]:
from tensorflow import keras
from tensorflow.keras import layers

def build_e2e_cnn(input_shape, num_classes):
    """Build and compile an end-to-end CNN model."""
    model = keras.Sequential([
        layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

## Task 2: Train the end-to-end CNN

Now implement `train_e2e_cnn` that trains your model on Fashion-MNIST.

Requirements:
- Split the training set into train/validation using `validation_split` or manual split.
- Train for **at least 3 epochs**.
- Return the **History** object from `model.fit`.

Do **not** hard-code global variables inside the function body (use arguments).

In [15]:
def train_e2e_cnn(model, x_train, y_train, epochs=3, batch_size=64, validation_split=0.1):
    """Train the end-to-end CNN model on Fashion-MNIST."""
    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        verbose=1
    )
    return history

## Task 3: Decide if a pipeline is end-to-end

Implement the function `is_end_to_end` that decides whether a given pipeline is **end-to-end deep learning**.

The pipeline is described by three boolean flags:
- `has_raw_input_to_model`: `True` if the model takes raw data (e.g. pixels, waveforms) as input, not hand-crafted features.
- `has_separate_manual_feature_stage`: `True` if there is a separate, hand-written feature extraction stage **outside** the neural network.
- `has_multiple_independent_models`: `True` if the pipeline consists of several independently trained models chained together (not one jointly trained model).

Rules (for this homework):
- A pipeline is **end-to-end** if:
  - `has_raw_input_to_model` is `True`, AND
  - `has_separate_manual_feature_stage` is `False`, AND
  - `has_multiple_independent_models` is `False`.
- Otherwise it is **not** end-to-end.

Return `True` if the pipeline is end-to-end, otherwise `False`.

In [17]:
def is_end_to_end(has_raw_input_to_model: bool,
                  has_separate_manual_feature_stage: bool,
                  has_multiple_independent_models: bool) -> bool:
    """Return True if the pipeline is end-to-end deep learning."""
    return has_raw_input_to_model and not has_separate_manual_feature_stage and not has_multiple_independent_models